# PySpark reference 14: Structured Streaming

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-reference-14-structured-streaming.ipynb)

Streaming sources and sinks, triggers, output modes, checkpoints and exactly-once, event-time windows and watermarks, deduplication, stream joins, arbitrary state, `foreachBatch` upserts, monitoring and Kafka. Practise it in [PySpark practice 14: Structured Streaming](pyspark-practice-14-structured-streaming.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and small helpers: `plan_lines(df)` returns what `explain()` prints, `final_plan(df)` runs a query and returns the plan Spark actually used, and `expected_error()` prints the error class of an example that fails on purpose. Helpers: `land(dir, rows)` drops a JSON Lines batch into a landing folder, `run_available_now(writer)` runs a streaming write until it has processed everything available, and `new_dir(name)` makes a fresh folder.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:110])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

import json
import os
import uuid

EVENT_SCHEMA = "event_id int, user_id int, event string, amount double, ts timestamp"


def new_dir(name):
    """A fresh directory under WORK_DIR, so every example starts clean."""
    path = os.path.join(WORK_DIR, f"{name}-{uuid.uuid4().hex[:6]}")
    os.makedirs(path)
    return path


def land(directory, rows, name=None):
    """Write one JSON Lines file into a landing directory, like a producer dropping a batch."""
    name = name or f"batch-{uuid.uuid4().hex[:8]}.json"
    with open(os.path.join(directory, name), "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")


def run_available_now(writer):
    """Start a streaming write that processes everything available, then stops; return the query."""
    query = writer.trigger(availableNow=True).start()
    query.awaitTermination()
    return query


def ev(event_id, user_id, event, amount, ts):
    return {"event_id": event_id, "user_id": user_id, "event": event, "amount": amount, "ts": ts}

## Contents

| section | entries |
| --- | --- |
| [The streaming model](#the-streaming-model) | [Streams as unbounded tables](#streams-as-unbounded-tables), [`spark.readStream`: `DataStreamReader`](#sparkreadstream-datastreamreader), [`df.writeStream`: `DataStreamWriter`](#dfwritestream-datastreamwriter), [Triggers](#triggers), [Output modes](#output-modes), [Checkpoints and exactly-once](#checkpoints-and-exactly-once) |
| [Event time and late data](#event-time-and-late-data) | [Event-time windows](#event-time-windows), [`withWatermark(eventTime, delayThreshold)`](#withwatermarkeventtime-delaythreshold) |
| [Stateful operations](#stateful-operations) | [Deduplication: `dropDuplicates` and `dropDuplicatesWithinWatermark`](#deduplication-dropduplicates-and-dropduplicateswithinwatermark), [Stream-static and stream-stream joins](#stream-static-and-stream-stream-joins), [Arbitrary state: `transformWithStateInPandas` and `applyInPandasWithState`](#arbitrary-state-transformwithstateinpandas-and-applyinpandaswithstate) |
| [foreachBatch](#foreachbatch) | [`writeStream.foreachBatch(func)`](#writestreamforeachbatchfunc) |
| [Monitoring and operations](#monitoring-and-operations) | [`StreamingQuery` status and progress](#streamingquery-status-and-progress), [Reading from and writing to Kafka](#reading-from-and-writing-to-kafka) |

## The streaming model

### Streams as unbounded tables

Structured Streaming treats a stream as a table that keeps growing. You write the same DataFrame code as for batch;
Spark runs it **incrementally** in micro-batches, processing only new input each time and keeping the state needed
for aggregations, deduplication and joins between batches.

| concept | meaning |
| --- | --- |
| source | where new data comes from: files, Kafka, rate, Delta tables |
| micro-batch | one incremental run over new input |
| trigger | when micro-batches run |
| output mode | which result rows are written each batch |
| sink | where results go: files, Delta, Kafka, memory, `foreachBatch` |
| checkpoint | offsets and state saved so a restarted query continues exactly where it stopped |

**Notes:** `df.isStreaming` tells a streaming DataFrame from a batch one. Actions such as `count()` or `show()` are not
allowed on a streaming DataFrame; results come out through `writeStream`.

### `spark.readStream`: `DataStreamReader`

| source | example | key options |
| --- | --- | --- |
| files | `spark.readStream.schema(s).json(dir)` | a schema is **required**; `maxFilesPerTrigger`, `maxBytesPerTrigger`, `latestFirst`, `cleanSource` |
| Kafka | `spark.readStream.format("kafka")` | `kafka.bootstrap.servers`, `subscribe`, `startingOffsets`, `maxOffsetsPerTrigger` |
| Delta | `spark.readStream.format("delta").load(path)` / `.table(name)` | `maxFilesPerTrigger`, `readChangeFeed` |
| rate | `spark.readStream.format("rate").option("rowsPerSecond", 10)` | generates `timestamp, value` rows for testing |

**Returns:** a streaming DataFrame.

**Notes:** a file source processes each file once, tracking processed files in the checkpoint. Producers must write
files atomically (write elsewhere, then move them in), or a half-written file can be read.

In [ ]:
landing = new_dir("landing")
land(landing, [ev(1, 1, "view", None, "2026-01-06 09:00:00"), ev(2, 1, "purchase", 30.0, "2026-01-06 09:10:00")])
stream = spark.readStream.schema(EVENT_SCHEMA).json(landing)
print("isStreaming:", stream.isStreaming)
with expected_error():
    stream.count()

### `df.writeStream`: `DataStreamWriter`

| method | description |
| --- | --- |
| `.format(name)` | sink: `"parquet"`, `"json"`, `"delta"`, `"kafka"`, `"memory"`, `"console"`, `"noop"` |
| `.outputMode(mode)` | `"append"`, `"update"` or `"complete"` |
| `.option("checkpointLocation", path)` | where offsets and state are saved (required for real sinks) |
| `.trigger(...)` | when to run micro-batches (below) |
| `.queryName(name)` | name in the UI and the table name for the memory sink |
| `.partitionBy(*cols)` | partition file sinks |
| `.foreachBatch(func)` / `.foreach(writer)` | custom write logic per batch or per row |
| `.start(path=None)` / `.toTable(name)` | start the query; returns a `StreamingQuery` |

**Returns:** a `StreamingQuery`: `awaitTermination()`, `stop()`, `status`, `lastProgress`, `recentProgress`, `id`,
`runId`.

In [ ]:
out, chk = new_dir("events_out"), new_dir("events_chk")
query = run_available_now(stream.writeStream.format("parquet").option("path", out).option("checkpointLocation", chk).outputMode("append"))
print("rows written:", spark.read.parquet(out).count(), "| input rows in last batch:", query.lastProgress["numInputRows"])

### Triggers

| trigger | behaviour | use for |
| --- | --- | --- |
| none (default) | next micro-batch as soon as the previous one finishes | low latency |
| `.trigger(processingTime="1 minute")` | a micro-batch every interval | steady pipelines |
| `.trigger(availableNow=True)` | process everything available now, in several batches if needed, then **stop** | scheduled incremental jobs (run by Airflow every hour) |
| `.trigger(once=True)` | one batch with everything, then stop (deprecated; use `availableNow`) | legacy jobs |
| `.trigger(continuous="1 second")` | experimental low-latency mode for simple map-only queries | rare |

**Notes:** `availableNow` turns a streaming query into an incremental batch job with exactly-once bookkeeping from the
checkpoint: each run picks up only what arrived since the last run.

In [ ]:
land(landing, [ev(3, 2, "view", None, "2026-01-06 10:00:00")])
query = run_available_now(stream.writeStream.format("parquet").option("checkpointLocation", chk).option("path", out))
print("second run read:", query.lastProgress["numInputRows"], "new row(s); table now has", spark.read.parquet(out).count(), "rows")

### Output modes

| mode | writes each batch | allowed for |
| --- | --- | --- |
| `append` | only rows that are final and will never change | queries without aggregation; aggregations **with** a watermark (rows are emitted once their window closes) |
| `update` | rows that changed in this batch | aggregations (with or without a watermark) |
| `complete` | the whole result table | aggregations only; state grows forever without a watermark |

**Notes:** file sinks support only `append`. Use `update` with `foreachBatch` (to merge) or a sink that handles upserts.

In [ ]:
counts = stream.groupBy("event").count()
q = run_available_now(counts.writeStream.format("memory").queryName("event_counts").outputMode("complete").option("checkpointLocation", new_dir("chk")))
spark.table("event_counts").orderBy("event").show()
with expected_error():
    counts.writeStream.format("parquet").option("path", new_dir("x")).option("checkpointLocation", new_dir("chk")).outputMode("append").start()

### Checkpoints and exactly-once

The checkpoint directory stores the source offsets of each batch, which batches committed, and the state of stateful
operators. On restart with the same checkpoint, the query re-runs an uncommitted batch and continues; with
**idempotent or transactional sinks** (files, Delta, `foreachBatch` keyed by batch id) the end result is
**exactly-once**.

**Notes**
- One checkpoint per query; never share or copy one between queries.
- Changing the query's stateful logic (grouping keys, aggregations) usually requires a new checkpoint.
- Deleting the checkpoint reprocesses all available input.

## Event time and late data

### Event-time windows

Group by `F.window(event_time_col, duration, slide)` to count per time bucket based on when events **happened**, not
when they arrived. Windows can be tumbling (no slide) or sliding (overlapping).

In [ ]:
win_land = new_dir("win_landing")
land(win_land, [ev(1, 1, "view", None, "2026-01-06 09:01:00"), ev(2, 2, "view", None, "2026-01-06 09:07:00"), ev(3, 1, "buy", 9.0, "2026-01-06 09:12:00")])
windowed = spark.readStream.schema(EVENT_SCHEMA).json(win_land).groupBy(F.window("ts", "10 minutes")).count()
run_available_now(windowed.writeStream.format("memory").queryName("win10").outputMode("complete").option("checkpointLocation", new_dir("chk")))
spark.table("win10").select("window.start", "window.end", "count").orderBy("start").show()

### `withWatermark(eventTime, delayThreshold)`

A watermark tells Spark how late data may arrive: `withWatermark("ts", "10 minutes")` means the query tracks the max event
time seen and treats anything older than `max - 10 minutes` as too late. It lets Spark **drop old state** and, in
`append` mode, **emit** a window once the watermark passes its end.

| parameter | type | description |
| --- | --- | --- |
| `eventTime` | str | the timestamp column |
| `delayThreshold` | str | how late data may be, e.g. `"10 minutes"` |

**Notes:** events later than the watermark are dropped from aggregations. A larger delay keeps more state and emits
results later; choose it from the real lateness of your sources. The watermark advances between micro-batches, based on
the previous batch's max event time.

In [ ]:
wm_land, wm_out, wm_chk = new_dir("wm_landing"), new_dir("wm_out"), new_dir("wm_chk")
wm_query_df = (spark.readStream.schema(EVENT_SCHEMA).json(wm_land).withWatermark("ts", "5 minutes")
               .groupBy(F.window("ts", "10 minutes")).agg(F.count("*").alias("events")))
writer = lambda: wm_query_df.writeStream.format("parquet").option("path", wm_out).option("checkpointLocation", wm_chk).outputMode("append")
land(wm_land, [ev(1, 1, "view", None, "2026-01-06 09:01:00"), ev(2, 1, "view", None, "2026-01-06 09:22:00")])
run_available_now(writer())
land(wm_land, [ev(3, 2, "view", None, "2026-01-06 09:03:00"), ev(4, 2, "view", None, "2026-01-06 09:40:00")])  # 09:03 is now too late
run_available_now(writer())
spark.read.parquet(wm_out).select("window.start", "events").orderBy("start").show()

## Stateful operations

### Deduplication: `dropDuplicates` and `dropDuplicatesWithinWatermark`

`dropDuplicates(["event_id"])` on a stream remembers every key ever seen, so state grows without bound unless a watermark
column is part of the key. `dropDuplicatesWithinWatermark(["event_id"])` (Spark 3.5+) keeps each key only until the
watermark passes, which bounds state while still removing retries that arrive within the delay.

In [ ]:
dd_land = new_dir("dd_landing")
land(dd_land, [ev(1, 1, "view", None, "2026-01-06 09:00:00"), ev(1, 1, "view", None, "2026-01-06 09:00:00"), ev(2, 2, "buy", 5.0, "2026-01-06 09:05:00")])
deduped = spark.readStream.schema(EVENT_SCHEMA).json(dd_land).withWatermark("ts", "1 hour").dropDuplicatesWithinWatermark(["event_id"])
run_available_now(deduped.writeStream.format("memory").queryName("deduped").outputMode("append").option("checkpointLocation", new_dir("chk")))
print("rows after dedup:", spark.table("deduped").count())

### Stream-static and stream-stream joins

| join | supported | notes |
| --- | --- | --- |
| stream ⋈ static (inner, left outer with stream on the left) | yes | the static side is re-read each batch if it is a file or table source |
| stream ⋈ stream inner | yes | add watermarks on both sides and a time-range condition to bound state |
| stream ⋈ stream outer | yes, with watermarks and a time condition | unmatched rows are emitted when the watermark passes |

**Use it for:** enriching events with a dimension (stream-static); matching impressions to clicks within 30 minutes
(stream-stream).

In [ ]:
enriched = spark.readStream.schema(EVENT_SCHEMA).json(landing).join(customers.withColumnRenamed("customer_id", "user_id"), "user_id", "left")
run_available_now(enriched.writeStream.format("memory").queryName("enriched").outputMode("append").option("checkpointLocation", new_dir("chk")))
spark.table("enriched").select("event_id", "user_id", "name", "region").orderBy("event_id").show()

### Arbitrary state: `transformWithStateInPandas` and `applyInPandasWithState`

For logic the built-in operators cannot express (per-user state machines, custom session rules, alerts after N events),
Spark offers per-key state with timeouts:

```python
def update(key, pdfs, state):                 # applyInPandasWithState (Spark 3.4+)
    count = state.get[0] if state.exists else 0
    for pdf in pdfs:
        count += len(pdf)
    state.update((count,))
    yield pd.DataFrame({"user_id": [key[0]], "events": [count]})

(stream.groupBy("user_id")
 .applyInPandasWithState(update, outputStructType="user_id int, events long",
                         stateStructType="events long", outputMode="update", timeoutConf="NoTimeout"))
```

Spark 4 adds `transformWithStateInPandas`, a richer API with multiple state variables, TTLs and timers.

**Notes:** keep state small and set timeouts or TTLs, or state grows forever; prefer built-in aggregations and windows.

## foreachBatch

### `writeStream.foreachBatch(func)`

Calls `func(batch_df, batch_id)` for every micro-batch with an ordinary **batch** DataFrame, so any batch API works:
`MERGE` into Delta, writes to several sinks, JDBC writes, deduplication against the target.

**Use it for:** upserts (stream → Delta `MERGE`), writing one stream to several tables, sinks without native streaming
support.

**Notes:** a batch can be re-run after a failure with the same `batch_id`; make the function idempotent (merge on keys, or
skip a `batch_id` already recorded). Cache `batch_df` if you write it to more than one place.

In [ ]:
target_dir = new_dir("latest_by_user")
seen_batches = []

def upsert(batch_df, batch_id):
    seen_batches.append(batch_id)
    latest = batch_df.withColumn("rn", F.row_number().over(Window.partitionBy("user_id").orderBy(F.desc("ts")))).where("rn = 1").drop("rn")
    try:
        current = spark.read.parquet(target_dir)
        merged = current.join(latest, "user_id", "left_anti").unionByName(latest)
    except Exception:
        merged = latest
    merged.cache().count()
    merged.write.mode("overwrite").parquet(target_dir + "_tmp")
    spark.read.parquet(target_dir + "_tmp").write.mode("overwrite").parquet(target_dir)

fb_land = new_dir("fb_landing")
land(fb_land, [ev(1, 1, "view", None, "2026-01-06 09:00:00"), ev(2, 2, "view", None, "2026-01-06 09:01:00")])
fb_chk = new_dir("fb_chk")
run_available_now(spark.readStream.schema(EVENT_SCHEMA).json(fb_land).writeStream.foreachBatch(upsert).option("checkpointLocation", fb_chk))
land(fb_land, [ev(3, 1, "purchase", 20.0, "2026-01-06 09:30:00")])
run_available_now(spark.readStream.schema(EVENT_SCHEMA).json(fb_land).writeStream.foreachBatch(upsert).option("checkpointLocation", fb_chk))
print("batches:", seen_batches)
spark.read.parquet(target_dir).select("user_id", "event_id", "event").orderBy("user_id").show()

## Monitoring and operations

### `StreamingQuery` status and progress

| member | returns |
| --- | --- |
| `query.status` | whether data is available, the trigger is active, a message |
| `query.lastProgress` / `query.recentProgress` | per-batch metrics: `numInputRows`, `inputRowsPerSecond`, `processedRowsPerSecond`, `durationMs`, `stateOperators`, `sources`, `sink` |
| `query.exception()` | the error that stopped the query, if any |
| `spark.streams.active`, `spark.streams.get(id)` | all running queries |
| `spark.streams.addListener(listener)` | a `StreamingQueryListener` called on start, progress and termination (push metrics to monitoring) |

**Notes:** alert when `processedRowsPerSecond` stays below `inputRowsPerSecond` (the query is falling behind) or when state
size keeps growing (`stateOperators[].numRowsTotal`).

In [ ]:
progress = q.lastProgress
print({k: progress[k] for k in ["batchId", "numInputRows"]}, "| state rows:", [s["numRowsTotal"] for s in progress.get("stateOperators", [])])
print("active queries:", len(spark.streams.active))

### Reading from and writing to Kafka

```python
raw = (spark.readStream.format("kafka")
       .option("kafka.bootstrap.servers", "broker:9092")
       .option("subscribe", "orders")                 # or "subscribePattern", "assign"
       .option("startingOffsets", "earliest")         # only for a new checkpoint
       .option("maxOffsetsPerTrigger", 100_000)       # cap batch size
       .load())
orders_stream = raw.select(F.from_json(F.col("value").cast("string"), ORDER_SCHEMA).alias("o"), "timestamp").select("o.*", "timestamp")

(orders_stream.select(F.col("order_id").cast("string").alias("key"), F.to_json(F.struct("*")).alias("value"))
 .writeStream.format("kafka")
 .option("kafka.bootstrap.servers", "broker:9092").option("topic", "orders_clean")
 .option("checkpointLocation", "/chk/orders_clean").start())
```

Kafka rows carry `key`, `value` (binary), `topic`, `partition`, `offset`, `timestamp` and `headers`. Offsets are tracked in
the checkpoint, not in Kafka consumer groups. The Kafka connector needs the `spark-sql-kafka-0-10` package.

**Next:** practise these entries in [PySpark practice 14: Structured Streaming](pyspark-practice-14-structured-streaming.ipynb).